In [8]:
from scipy.stats import norm
import numpy as np
import time
import import_ipynb
from Geometric_Brownian_Motion import simulate_GBM, simulate_GBM_antithetic
from Black_Scholes_Closed_Form import european_mc, european_mc_antithetic
from Longstaff_Schwartz_American_Options import american_longstaff_schwartz
#from Greeks import pathwise_delta, pathwise_vega, bs_delta, bs_vega
import matplotlib.pyplot as plt
from scipy.stats import qmc

American put: 6.0859 ± 0.0440
European put: 5.5577
American call: 10.4120 ± 0.0910
European call: 10.4442


In [9]:
S0, K, r, sigma, T = 100, 100, 0.05, 0.2, 1.0

In [10]:
def american_binomial(S0, K, r, sigma, T, n, type="put"):
    dt = T / n
    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u
    p = (np.exp(r * dt) - d) / (u - d)
    discount = np.exp(-r * dt)
    
    # terminal stock prices at each node, step n_steps
    j = np.arange(n + 1)
    S_terminal = S0 * (u ** (n - j)) * (d ** j)
    
    if type == "put":
        values = np.maximum(K - S_terminal, 0)
    else:
        values = np.maximum(K - S_terminal, 0)  # placeholder, fixed below
        values = np.maximum(S_terminal - K, 0)
    
    # walk backward
    for step in range(n - 1, -1, -1):
        j = np.arange(step + 1)
        S_t = S0 * (u ** (step - j)) * (d ** j)
        
        continuation = discount * (p * values[:step+1] + (1-p) * values[1:step+2])
        
        if type == "put":
            exercise = np.maximum(K - S_t, 0)
        else:
            exercise = np.maximum(S_t - K, 0)
        
        values = np.maximum(continuation, exercise)
    
    return values[0]

In [11]:
am_put_price, am_se = american_longstaff_schwartz(S0, K, r, sigma, T, n=252, n_paths=100000, type="put")

In [12]:
binomial_put_price = american_binomial(S0, K, r, sigma, T, n=1000, type="put")

print(f"Longstaff-Schwartz American put: {am_put_price:.4f} ± {1.96*am_se:.4f}")
print(f"Binomial tree American put:      {binomial_put_price:.4f}")

Longstaff-Schwartz American put: 6.0513 ± 0.0435
Binomial tree American put:      6.0896
